> **Historical analysis:** This notebook includes a Brand-to-J&J normalization that has since been withdrawn. Its saved results describe that earlier assumption, not the current cleaned dataset. Do not use those results as current manufacturer evidence. See [current decisions](../docs/decisions.md) and revalidate affected findings before reuse.

# What does the Platform column represent?

Explore the supplied training and target data to infer the role of `Platform` from observed labels and product context. This is an empirical interpretation, not a confirmed Product Owner definition. Source CSV files are read only. The analysis copy applies the user-confirmed J&J manufacturer normalization using notebooks/references/historical_jj_brands.json; Platform values remain unchanged.


## Main finding: missing Platform within manufacturer-consistent J&J records

Using the validated J&J brand whitelist and the confirmed manufacturer normalization, there are **1,079 J&J records: 1,030 with Platform and 49 without it** (95.46% coverage). All 49 missing-Platform records have `Mnfr = J&J` in this analysis. Investigate these as possible incomplete Platform classification within the J&J portfolio.

The supplied raw CSV retains earlier manufacturer assignments. Those source assignments are preserved separately for provenance and do not describe the corrected analysis state. Platform remains concentrated on J&J and selected other brands; the business reason for that scope still needs owner confirmation.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'provided').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'provided'
train = pd.read_csv(DATA / 'Selfcare_Training_data (1).csv', low_memory=False)
target = pd.read_csv(DATA / 'Selfcare_Target_data (1).csv', low_memory=False)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 180)
print('Training shape:', train.shape)
print('Target shape:', target.shape)

# Reproduce the user-confirmed manufacturer normalization on an analysis copy.
# Preserve source assignments for provenance; do not modify the supplied CSV.
import json
jj_whitelist = set(json.loads((ROOT / 'notebooks' / 'references' / 'historical_jj_brands.json').read_text()))
raw_mnfr = train['Mnfr'].copy()
train.loc[train['Brand'].isin(jj_whitelist), 'Mnfr'] = 'J&J'
print('Analysis uses validated J&J brand-to-manufacturer normalization.')


Training shape: (84552, 32)
Target shape: (28082, 32)
Analysis uses validated J&J brand-to-manufacturer normalization.


## 1. How often is Platform populated?

In [2]:
summary = pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'labeled': df['Platform'].notna().sum(),
     'missing_pct': df['Platform'].isna().mean() * 100,
     'unique_labels': df['Platform'].nunique()}
    for name, df in [('training', train), ('target', target)]
])
print(summary.to_string(index=False))
labeled = train.loc[train['Platform'].notna()].copy()
print('\nAll observed labels (including potentially malformed records):')
print(labeled['Platform'].value_counts().to_string())


 dataset  rows  labeled  missing_pct  unique_labels
training 84552     1105    98.693112             63
  target 28082        0   100.000000              0

All observed labels (including potentially malformed records):
Platform
Extra Strength                                                                                          84
Children                                                                                                72
Pain + Fever                                                                                            51
Oral Suspension                                                                                         49
Motrin IB                                                                                               47
24 Hour Relief                                                                                          46
Eight Hour Arthritis Pain                                                                               43
Throat                

## 2. What products do the labels describe?
Compare each label with its brand, category hierarchy, and product name. Multiple examples help expose variation and annotation inconsistencies.

In [3]:
context = ['Brand', 'Platform', 'Segment', 'Sub-Segment', 'ProductName']
examples = labeled.sort_values(['Platform', 'Brand']).groupby('Platform', sort=True).head(2)
print(examples[context].to_string(index=False, max_colwidth=100))


             Brand                                                                                             Platform                             Segment                     Sub-Segment                                                                                          ProductName
       gluten-free                                                                                            grass fed  IGEN Non-GMO tested.* No soy wheat                         lactose                                                                                                  NaN
       gluten-free                                                                                            grass fed  IGEN Non-GMO tested.* No soy wheat                         lactose Serving Size 3 Tablets. Calories: 10; Protein: 3 g; Vitamin C (as calcium ascorbate): 30 mg; Biot...
               tea  orange juice or a smoothie daily; unflavored GRASS-FED: Super Collagen Peptides powder is grass-fed              

## 3. Is Platform specific to a brand or retailer?
Coverage is the percentage of rows with a nonmissing Platform in each group. A nonmissing value is not necessarily a valid label.

In [4]:
for col in ['Mnfr', 'Brand', 'Retailer', 'Segment']:
    coverage = train.groupby(col, dropna=False)['Platform'].agg(
        rows='size', labeled='count', unique_platforms='nunique')
    coverage['coverage_pct'] = 100 * coverage['labeled'] / coverage['rows']
    print('\nCoverage by', col, '(top 25 groups by labeled count)')
    print(coverage.sort_values('labeled', ascending=False).head(25).to_string())

print('\nPlatforms used by multiple brands:')
brand_links = labeled.groupby('Platform')['Brand'].agg(lambda s: sorted(set(s.dropna())))
print(brand_links[brand_links.map(len) > 1].to_string())



Coverage by Mnfr (top 25 groups by labeled count)
                                                                                                         rows  labeled  unique_platforms  coverage_pct
Mnfr                                                                                                                                                  
J&J                                                                                                      1079     1030                55     95.458758
All others                                                                                              77816       72                10      0.092526
 aids in collagen formation* GRASS-FED COLLAGEN: Super Collagen + Vitamin C & Biotin is Keto certified      2        2                 1    100.000000
 coffee                                                                                                     1        1                 1    100.000000
NaN                                        

## 4. Do labels resemble product-name attributes?
Literal matching is a diagnostic only: spelling, punctuation, synonyms, and incomplete names can prevent a match. It does not validate or invalidate an annotation.

In [5]:
has_name = labeled['ProductName'].notna()
name_rows = labeled.loc[has_name]
literal_match = name_rows.apply(
    lambda row: str(row['Platform']).strip().casefold() in str(row['ProductName']).casefold(), axis=1)
print('Labeled rows with a product name:', len(name_rows))
print('Platform occurs literally in product name:', int(literal_match.sum()))
print('Literal match percentage:', round(literal_match.mean() * 100, 2))
print('\nExamples without a literal match (not automatically errors):')
print(name_rows.loc[~literal_match, context].head(15).to_string(index=False, max_colwidth=100))


Labeled rows with a product name: 1095
Platform occurs literally in product name: 686
Literal match percentage: 62.65

Examples without a literal match (not automatically errors):
   Brand                  Platform             Segment               Sub-Segment                                                                                    ProductName
  Visine                   Red Eye     Other Self Care                  Eye Care                 Visine Advanced Relief Redness Reliever Eye Drops, 0.28 Fluid Ounce Pack of 12
  Pepcid          Maximum Strength    Digestive Health               Acid Relief                                                          McNeil PPC Pepcid Acid Reducer, 50 ea
 Tylenol Eight Hour Arthritis Pain Internal Analgesics           Speciality Pain                         Tylenol 8 HR Arthritis Pain Extended Release Caplets, 650 Mg (290 ct.)
  Motrin                 Motrin IB Internal Analgesics             Internal Pain                          Infants' M

## 5. Inspect potentially malformed records
Leading whitespace, unusually long labels, and missing product names are review signals, not automatic deletion rules. The 80-character threshold is exploratory.

In [6]:
platform = labeled['Platform'].astype('string')
suspicious = (platform.ne(platform.str.strip()) | platform.str.len().gt(80)
              | labeled['ProductName'].isna())
print('Flagged rows:', int(suspicious.sum()))
print(labeled.loc[suspicious, ['Retailer'] + context].to_string(max_colwidth=120))


Flagged rows: 11
                                                                     Retailer         Brand                                                                                              Platform                              Segment         Sub-Segment                                                                                                              ProductName
13599                                                                  Kroger       Tylenol                                                                             Eight Hour Arthritis Pain                  Internal Analgesics     Speciality Pain                                                                                                                      NaN
16557                                                                  Kroger      Zarbee's                                                                                                Throat                                 CCFS         

## Interpretation and modeling implications

`Platform` appears to describe a **product line or variant within a brand**, with mixed types of attributes:

- Strength: `Extra Strength`, `Regular Strength`, `Maximum Strength`.
- Form: `Liquid Gels`, `Chewables`, `Cream`, `Oral Suspension`.
- Benefit or condition: `Pain + Fever`, `Cold + Flu`, `Dry Eye`.
- Audience or duration: `Children`, `24 Hour Relief`.
- Named product lines: `Motrin IB`, `Tylenol PM`, `Imodium A-D`.

For example, Brand = Tylenol with Platform = Extra Strength identifies a more specific product variant. Platform is not a single consistent attribute such as dosage form, and the observed labels do not describe an ecommerce platform. `Retailer` is recorded separately.

Only 1,105 of 84,552 training records have a Platform value (98.69% missing); all 28,082 target records are missing it. Some populated values contain apparent description fragments and occur on suspicious records. Nonmissing counts therefore overstate the number of trustworthy labels.

The repository lists Platform as a prediction target. Preserve missing values as unlabeled pending clarification; do not infer that missing means not applicable. Review suspect records before training, and inspect coverage by brand before assuming a model will generalize across the catalog. Product-name matches are supporting evidence, not ground truth.

**Questions for the Product Owner:** Is Platform formally a brand-specific product-line taxonomy? Which brands require it? Can multiple attributes apply, and which takes precedence (for example, Tylenol PM versus Extra Strength)? Does a blank mean unannotated or not applicable?


## 6. J&J Platform coverage after manufacturer normalization

Use the validated whitelist in `notebooks/references/historical_jj_brands.json` and the user-confirmed manufacturer assignments. These are dataset classification rules, not a statement of current corporate ownership.


In [7]:
jj = train['Mnfr'].eq('J&J')
jj_brands = sorted(jj_whitelist)
associated = train['Brand'].isin(jj_brands)
missing_associated = train.loc[associated & train['Platform'].isna()].copy()
print('Associated brands:', jj_brands)
assert missing_associated['Mnfr'].eq('J&J').all()
print('Normalized J&J rows:', int(jj.sum()))
print('Normalized J&J rows missing Platform:', int(train.loc[jj, 'Platform'].isna().sum()))
print('Associated-brand rows missing Platform:', len(missing_associated))
print('\nManufacturer values among those missing rows:')
print(missing_associated['Mnfr'].value_counts(dropna=False).to_string())
print('\nMissing target fields by cohort:')
target_columns = ['Mnfr', 'Brand', 'Platform', 'Segment', 'Sub-Segment', 'TargetAgeGroup']
print(pd.DataFrame({
    'normalized_JJ': train.loc[jj, target_columns].isna().sum(),
    'associated_brand_missing_platform': missing_associated[target_columns].isna().sum()
}).to_string())
print('\nAssociated-brand coverage by brand:')
print(train.loc[associated].groupby('Brand')['Platform'].agg(rows='size', labeled='count').to_string())


Associated brands: ['Benadryl', 'Bengay', 'Imodium', 'Motrin', 'Pepcid', 'Sudafed', 'Tylenol', 'Visine', "Zarbee's", 'Zyrtec']
Normalized J&J rows: 1079
Normalized J&J rows missing Platform: 49
Associated-brand rows missing Platform: 49

Manufacturer values among those missing rows:
Mnfr
J&J    49

Missing target fields by cohort:
                normalized_JJ  associated_brand_missing_platform
Mnfr                        0                                  0
Brand                       0                                  0
Platform                   49                                 49
Segment                    30                                 30
Sub-Segment                49                                 49
TargetAgeGroup             49                                 49

Associated-brand coverage by brand:
          rows  labeled
Brand                  
Benadryl   129      119
Bengay      34       32
Imodium     45       45
Motrin     108      104
Pepcid      92       87
Sudafed

## 7. Why is Platform missing on these 49 J&J rows?

All 49 records have the consistent manufacturer assignment J&J after normalization. Their missing Platform values should be reviewed as possible incomplete classification. All also lack Sub-Segment and TargetAgeGroup in the supplied data. Product-name anomalies can merit separate review but do not establish manufacturer inconsistency in the corrected records.


In [8]:
print(missing_associated[['Mnfr', 'Brand', 'Platform', 'Segment', 'Sub-Segment',
                          'TargetAgeGroup', 'Retailer', 'ProductName']].to_string(max_colwidth=120))
print('\nMissing cohort by retailer:')
print(missing_associated['Retailer'].value_counts().to_string())


      Mnfr     Brand Platform           Segment Sub-Segment TargetAgeGroup   Retailer                                                                                                              ProductName
291    J&J   Tylenol      NaN               NaN         NaN            NaN        CVS                                                     Tylenol Extra Strength Severe Cough + Sore Throat Night Liquid, 8 OZ
6730   J&J  Benadryl      NaN           Allergy         NaN            NaN    Walmart                                                         Benadryl Allergy Plus Congestion Ultra Tablets 24 ea (Pack of 4)
7005   J&J    Motrin      NaN               NaN         NaN            NaN     Target                                                       Motrin Acetaminophen Dual Action with Tylenol Pain Reliever - 20ct
7239   J&J    Pepcid      NaN  Digestive Health         NaN            NaN    Walmart                                         2 Pack Pepcid Complete Acid Reducer Antacid Tr

## 8. Do unlabeled products have labeled counterparts?

Match nonempty UPC or normalized exact product name within the same brand. Keep all distinct candidate labels so ambiguity remains visible. Names alone do not establish identical SKUs; these comparisons support review rather than automatic imputation.


In [9]:
available = train.loc[train['Platform'].notna()].copy()
for key in ['Upc', 'ProductName']:
    left = missing_associated.copy()
    right = available.copy()
    for frame in [left, right]:
        frame['_match_key'] = frame[key].astype('string').str.strip().str.casefold()
    left = left.loc[left['_match_key'].notna() & left['_match_key'].ne('')].reset_index(names='source_row')
    right = right.loc[right['_match_key'].notna() & right['_match_key'].ne('')]
    candidates = right.groupby(['Brand', '_match_key'])['Platform'].agg(lambda x: sorted(set(x)))
    matches = left.merge(candidates.rename('candidate_platforms'), on=['Brand', '_match_key'], how='inner')
    print('\nMatching by', key, 'and Brand:', matches['source_row'].nunique(), 'unlabeled rows')
    print(matches[['source_row', 'Brand', 'ProductName', 'candidate_platforms']].to_string(index=False, max_colwidth=120))



Matching by Upc and Brand: 1 unlabeled rows
 source_row  Brand                                                                               ProductName candidate_platforms
      78239 Motrin Childrens Motrin Pain Reliever/Fever Reducer Liquid - Ibuprofen (NSAID) - Berry - 4 fl oz   [Oral Suspension]

Matching by ProductName and Brand: 4 unlabeled rows
 source_row   Brand                                                            ProductName candidate_platforms
      11950  Zyrtec       Zyrtec 24 Hour Children's Allergy Chewable Tablets, Grape, 12 ct    [24 Hour Relief]
      30555 Sudafed     Sudafed PE Non-Drowsy Head Congestion + Pain Relief Caplets, 20 ct              [Head]
      40266  Pepcid               Pepcid AC Maximum Strength Famotidine 20 mg, 125 Tablets  [Maximum Strength]
      50377  Zyrtec Zyrtec 24 Hour Allergy Relief Tablets with 10 mg Cetirizine HCl, 90 ct    [24 Hour Relief]


## 9. Why are there non-J&J Platform labels?

Separate the ordinary All others manufacturer group from suspicious manufacturer text. Check how selectively even these other brands receive Platform.


In [10]:
other_labeled = train.loc[~jj & train['Platform'].notna()]
print('Non-J&J labeled rows:', len(other_labeled))
print(other_labeled.groupby(['Mnfr', 'Brand'], dropna=False).size().to_string())
other_brands = train.loc[train['Mnfr'].eq('All others') & train['Platform'].notna(), 'Brand'].unique()
print('\nCoverage across the other labeled brands:')
print(train.loc[train['Brand'].isin(other_brands)].groupby('Brand')['Platform'].agg(rows='size', labeled='count').to_string())
print('\nOther-brand label choices:')
print(train.loc[train['Mnfr'].eq('All others') & train['Platform'].notna()].groupby(['Brand', 'Platform']).size().to_string())


Non-J&J labeled rows: 75
Mnfr                                                                                                    Brand             
 aids in collagen formation* GRASS-FED COLLAGEN: Super Collagen + Vitamin C & Biotin is Keto certified   gluten-free           2
 coffee                                                                                                  tea                   1
All others                                                                                              Advil                 13
                                                                                                        Claritin               1
                                                                                                        Excedrin              52
                                                                                                        Kirkland Signature     4
                                                                              

## Answer: what we can and cannot explain

**Why J&J?** After the confirmed manufacturer normalization, 1,030 of 1,079 J&J rows have Platform. The taxonomy singles out J&J against All others, and Platform labels are heavily concentrated in that portfolio. This suggests a focal-portfolio classification scheme, but the data cannot establish the sponsor's business reason or annotation instructions.

**Why not the 49?** All 49 Platform-missing records belonging to validated J&J brands are manufacturer-consistent (`Mnfr = J&J`). Treat their blanks as possible incomplete Platform classification. All also lack Sub-Segment and TargetAgeGroup. Four have same-brand exact-name matches to Platform-labeled rows, and one has a same-brand UPC match. These are useful review candidates, not automatic fill rules. The reason for the missing annotations remains unconfirmed.

**Why some other brands?** The remaining populated values include 72 All others rows (Excedrin 52, Advil 13, Kirkland Signature 4, Claritin 1, UnItemised brand 2) and three suspicious records. Selective comparator annotation is plausible but unconfirmed.

**Next step:** Review the 49 within the J&J cohort for missing product-line classification. Preserve blanks until valid labels are established. Ask the owner whether Platform is intended for all validated J&J products plus selected comparator products, and request the applicability and labeling rules. Do not interpret these blanks as manufacturer inconsistency or confirmed non-applicability.
